# Malaria blood-cell image classifier

This notebook audits the image folders, derives labels only from `Parasitized` and `Uninfected` directory names, removes exact duplicate files, creates reproducible group-aware train/validation/test splits, trains a transfer-learning classifier, and evaluates it on the held-out test set with visual and numeric results.

**Label caveat:** these are inherited folder labels, not independently verified diagnoses. This is a learning/research demonstration, not a medical device or diagnostic tool. It cannot determine an individual's malaria risk; suspected malaria requires a qualified health professional and appropriate testing.

The split manifests and model/results are written under `artifacts\malaria_classifier`. Original image files are not moved or modified.

## 1. Dependencies

Run this once in the notebook kernel if imports below are missing. The pretrained ResNet-18 weights are downloaded on first use, so internet access is required unless the weights are already cached. Training uses CUDA when available and otherwise runs on CPU; CPU training can take substantially longer.

In [ ]:
%pip install torch torchvision scikit-learn matplotlib pandas pillow

In [ ]:
from pathlib import Path
from collections import Counter
import hashlib
import json
import re
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
import sklearn
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, roc_curve, auc,
                             precision_recall_curve, average_precision_score)
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

SEED = 42
IMAGE_SIZE = 224
BATCH_SIZE = 32
MAX_EPOCHS = 12
EARLY_STOPPING_PATIENCE = 3
NUM_WORKERS = 0  # Safe default for Windows notebooks.
CLASS_NAMES = ["Uninfected", "Parasitized"]
LABEL_TO_ID = {name: i for i, name in enumerate(CLASS_NAMES)}
IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}

ROOT = Path.cwd()
SOURCE_ROOTS = [ROOT / "data_set", ROOT / "cell_images"]
OUTPUT_DIR = ROOT / "artifacts" / "malaria_classifier"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Project root: {ROOT}")
print(f"Device: {DEVICE}; PyTorch: {torch.__version__}; scikit-learn: {sklearn.__version__}")
print(f"Output directory: {OUTPUT_DIR}")

## 2. Audit labels and remove exact duplicates

A file receives a class only when its path contains exactly one known class folder. Images anywhere else are counted as unlabeled and excluded. SHA-256 deduplication prevents copies in the nested folders from appearing in multiple splits. If identical image bytes have conflicting folder labels, execution stops for manual review.

In [ ]:
def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()

all_image_paths = []
for source_root in SOURCE_ROOTS:
    if source_root.exists():
        all_image_paths.extend(
            path for path in source_root.rglob("*")
            if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
        )
    else:
        print(f"Note: folder not found and skipped: {source_root}")

if not all_image_paths:
    raise FileNotFoundError(f"No supported image files found under {SOURCE_ROOTS}")

unlabeled_paths = []
physical_label_counts = Counter()
records_by_hash = {}
conflicting_hashes = []
invalid_images = []

for path in sorted(set(all_image_paths)):
    labels_in_path = set(path.parts).intersection(CLASS_NAMES)
    if len(labels_in_path) != 1:
        if not labels_in_path:
            unlabeled_paths.append(str(path))
            continue
        raise ValueError(f"Path contains more than one class directory: {path}")
    label = labels_in_path.pop()
    physical_label_counts[label] += 1
    try:
        with Image.open(path) as image:
            image.verify()
        image_hash = sha256_file(path)
    except Exception as exc:
        invalid_images.append({"path": str(path), "error": str(exc)})
        continue

    existing = records_by_hash.get(image_hash)
    if existing is not None and existing["label"] != label:
        conflicting_hashes.append((existing["path"], str(path)))
    elif existing is None:
        group_match = re.search(r"_cell_\d+$", path.stem, flags=re.IGNORECASE)
        group_id = path.stem[:group_match.start()] if group_match else path.stem
        records_by_hash[image_hash] = {
            "path": str(path), "label": label, "label_id": LABEL_TO_ID[label],
            "sha256": image_hash, "group_id": group_id
        }

if conflicting_hashes:
    raise ValueError(f"Found {len(conflicting_hashes)} identical image(s) with conflicting labels; inspect conflicting_hashes.")

if invalid_images:
    pd.DataFrame(invalid_images).to_csv(OUTPUT_DIR / "invalid_images.csv", index=False)
    print(f"Warning: skipped {len(invalid_images)} unreadable image(s); see invalid_images.csv")

records = list(records_by_hash.values())
df = pd.DataFrame(records).sort_values(["label", "path"]).reset_index(drop=True)
if set(df["label"].unique()) != set(CLASS_NAMES):
    raise ValueError(f"Both classes are required; found {df['label'].value_counts().to_dict()}")

print(f"Image files discovered: {len(all_image_paths)}")
print(f"Labeled files by folder name: {dict(physical_label_counts)}")
print(f"Unlabeled image files excluded: {len(unlabeled_paths)}")
print(f"Unreadable labeled images excluded: {len(invalid_images)}")
print(f"Exact duplicate files removed: {len(all_image_paths) - len(unlabeled_paths) - len(invalid_images) - len(df)}")
print(f"Unique labeled images after exact deduplication: {len(df)}")
print("Unique class counts:", df["label"].value_counts().reindex(CLASS_NAMES).to_dict())
print(f"Unique filename groups: {df['group_id'].nunique()}")
pd.DataFrame({"path": unlabeled_paths}).to_csv(OUTPUT_DIR / "unlabeled_images.csv", index=False)
if unlabeled_paths:
    print("Unlabeled examples:", unlabeled_paths[:10])
df.head()

## 3. Inspect the labeled images

In [ ]:
counts = df["label"].value_counts().reindex(CLASS_NAMES, fill_value=0)
fig, ax = plt.subplots(figsize=(6, 4))
counts.plot(kind="bar", ax=ax, color=["#4C9F70", "#D95F59"])
ax.set(title="Unique labeled images", xlabel="Folder-derived class", ylabel="Images")
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "class_counts.png", dpi=160, bbox_inches="tight")
plt.show()

sample_df = pd.concat([
    df[df.label == name].sample(min(6, int(counts[name])), random_state=SEED)
    for name in CLASS_NAMES
])
fig, axes = plt.subplots(2, 6, figsize=(15, 5))
for ax, row in zip(axes.flat, sample_df.itertuples()):
    with Image.open(row.path) as image:
        ax.imshow(image.convert("RGB"))
    ax.set_title(row.label, fontsize=9)
    ax.axis("off")
for ax in axes.flat[len(sample_df):]:
    ax.axis("off")
fig.suptitle("Example images (labels inherited from folder names)")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "sample_images.png", dpi=160, bbox_inches="tight")
plt.show()

## 4. Group-aware train / validation / test split

Images with the same filename prefix before `_cell_<number>` are kept together to reduce capture-level leakage. Stratified group folds target approximately 71/14/14; exact proportions can vary because entire groups must remain intact. The test set is held out from training and tuning.

In [ ]:
outer_split = StratifiedGroupKFold(n_splits=7, shuffle=True, random_state=SEED)
train_val_idx, test_idx = next(outer_split.split(df, df.label_id, groups=df.group_id))
train_val = df.iloc[train_val_idx].reset_index(drop=True)
test_df = df.iloc[test_idx].reset_index(drop=True)

inner_split = StratifiedGroupKFold(n_splits=6, shuffle=True, random_state=SEED + 1)
train_idx, val_idx = next(inner_split.split(train_val, train_val.label_id, groups=train_val.group_id))
train_df = train_val.iloc[train_idx].reset_index(drop=True)
val_df = train_val.iloc[val_idx].reset_index(drop=True)

def group_set(frame):
    return set(frame.group_id)

assert not (group_set(train_df) & group_set(val_df))
assert not (group_set(train_df) & group_set(test_df))
assert not (group_set(val_df) & group_set(test_df))
assert not (set(train_df.sha256) & set(val_df.sha256))
assert not (set(train_df.sha256) & set(test_df.sha256))
assert not (set(val_df.sha256) & set(test_df.sha256))

for split_name, split_df in [("train", train_df), ("validation", val_df), ("test", test_df)]:
    split_df.to_csv(OUTPUT_DIR / f"{split_name}_manifest.csv", index=False)

# Export a conventional split/class folder tree without modifying source images.
import shutil
for split_name, split_df in [("train", train_df), ("validation", val_df), ("test", test_df)]:
    for row in split_df.itertuples():
        destination_dir = OUTPUT_DIR / "dataset_split" / split_name / row.label
        destination_dir.mkdir(parents=True, exist_ok=True)
        destination = destination_dir / f"{row.sha256[:12]}_{Path(row.path).name}"
        shutil.copy2(row.path, destination)

split_summary = pd.DataFrame({
    "images": [len(train_df), len(val_df), len(test_df)],
    "groups": [train_df.group_id.nunique(), val_df.group_id.nunique(), test_df.group_id.nunique()],
    "Uninfected": [(x.label == "Uninfected").sum() for x in (train_df, val_df, test_df)],
    "Parasitized": [(x.label == "Parasitized").sum() for x in (train_df, val_df, test_df)]
}, index=["train", "validation", "test"])
display(split_summary)
print("Actual split fractions:", (split_summary.images / len(df)).round(3).to_dict())

## 5. Build the transfer-learning model

A pretrained ResNet-18 is fine-tuned with ImageNet normalization, mild training augmentation, AdamW, validation-loss learning-rate reduction, and early stopping. These choices optimize on the validation set only; the held-out test set is used for the final report.

In [ ]:
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(12),
    transforms.ColorJitter(brightness=0.08, contrast=0.08),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class BloodCellDataset(Dataset):
    def __init__(self, frame, transform):
        self.frame = frame.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        with Image.open(row.path) as image:
            image = image.convert("RGB")
        return self.transform(image), int(row.label_id), row.path

train_loader = DataLoader(BloodCellDataset(train_df, train_transform), batch_size=BATCH_SIZE,
                          shuffle=True, num_workers=NUM_WORKERS, pin_memory=DEVICE.type == "cuda")
val_loader = DataLoader(BloodCellDataset(val_df, eval_transform), batch_size=BATCH_SIZE,
                        shuffle=False, num_workers=NUM_WORKERS, pin_memory=DEVICE.type == "cuda")
test_loader = DataLoader(BloodCellDataset(test_df, eval_transform), batch_size=BATCH_SIZE,
                         shuffle=False, num_workers=NUM_WORKERS, pin_memory=DEVICE.type == "cuda")

weights = models.ResNet18_Weights.DEFAULT
model = models.resnet18(weights=weights)
model.fc = nn.Linear(model.fc.in_features, len(CLASS_NAMES))
model = model.to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.3, patience=1)
print(f"Train / validation / test images: {len(train_df):,} / {len(val_df):,} / {len(test_df):,}")
print(f"Model: ResNet-18 pretrained on ImageNet; parameters: {sum(p.numel() for p in model.parameters()):,}")

## 6. Train with validation-based checkpointing

The checkpoint with the best validation loss is restored after early stopping. No test-set metrics are used to choose epochs or model parameters.

In [ ]:
def run_epoch(loader, training):
    model.train(training)
    total_loss = 0.0
    total_correct = 0
    total = 0
    for images, labels, _ in loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        if training:
            optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(training):
            logits = model(images)
            loss = criterion(logits, labels)
            if training:
                loss.backward()
                optimizer.step()
        total_loss += loss.item() * labels.size(0)
        total_correct += (logits.argmax(dim=1) == labels).sum().item()
        total += labels.size(0)
    return total_loss / total, total_correct / total

history = []
best_val_loss = float("inf")
best_epoch = 0
epochs_without_improvement = 0
checkpoint_path = OUTPUT_DIR / "best_resnet18_malaria.pt"
start_time = time.time()

for epoch in range(1, MAX_EPOCHS + 1):
    train_loss, train_accuracy = run_epoch(train_loader, training=True)
    val_loss, val_accuracy = run_epoch(val_loader, training=False)
    scheduler.step(val_loss)
    current_lr = optimizer.param_groups[0]["lr"]
    history.append({"epoch": epoch, "train_loss": train_loss, "train_accuracy": train_accuracy,
                    "val_loss": val_loss, "val_accuracy": val_accuracy, "learning_rate": current_lr})
    print(f"Epoch {epoch:02d}/{MAX_EPOCHS} | lr {current_lr:.2g} | "
          f"train loss {train_loss:.4f}, acc {train_accuracy:.4f} | "
          f"val loss {val_loss:.4f}, acc {val_accuracy:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch
        epochs_without_improvement = 0
        torch.save({"model_state_dict": model.state_dict(), "class_names": CLASS_NAMES,
                    "image_size": IMAGE_SIZE, "mean": IMAGENET_MEAN, "std": IMAGENET_STD,
                    "best_epoch": best_epoch, "best_validation_loss": best_val_loss}, checkpoint_path)
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
            print("Early stopping: validation loss did not improve.")
            break

checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])
history_df = pd.DataFrame(history)
history_df.to_csv(OUTPUT_DIR / "training_history.csv", index=False)
print(f"Best epoch: {best_epoch}; best validation loss: {best_val_loss:.4f}")
print(f"Training time: {(time.time() - start_time) / 60:.1f} minutes")
print(f"Saved model: {checkpoint_path}")

## 7. Training curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history_df.epoch, history_df.train_loss, marker="o", label="train")
axes[0].plot(history_df.epoch, history_df.val_loss, marker="o", label="validation")
axes[0].set(title="Cross-entropy loss", xlabel="Epoch", ylabel="Loss")
axes[0].legend()
axes[1].plot(history_df.epoch, history_df.train_accuracy, marker="o", label="train")
axes[1].plot(history_df.epoch, history_df.val_accuracy, marker="o", label="validation")
axes[1].set(title="Accuracy", xlabel="Epoch", ylabel="Accuracy", ylim=(0, 1.02))
axes[1].legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "training_curves.png", dpi=160, bbox_inches="tight")
plt.show()

## 8. Final held-out test results

This is the first use of the test set for model evaluation. Metrics summarize this dataset split only and should not be interpreted as clinical performance.

In [ ]:
model.eval()
test_true, test_probabilities, test_paths = [], [], []
with torch.no_grad():
    for images, labels, paths in test_loader:
        probabilities = torch.softmax(model(images.to(DEVICE)), dim=1).cpu().numpy()
        test_true.extend(labels.numpy().tolist())
        test_probabilities.extend(probabilities.tolist())
        test_paths.extend(paths)

test_true = np.asarray(test_true)
test_probabilities = np.asarray(test_probabilities)
test_predicted = test_probabilities.argmax(axis=1)
test_results = pd.DataFrame({
    "path": test_paths,
    "actual": [CLASS_NAMES[i] for i in test_true],
    "predicted": [CLASS_NAMES[i] for i in test_predicted],
    "prob_uninfected": test_probabilities[:, LABEL_TO_ID["Uninfected"]],
    "prob_parasitized": test_probabilities[:, LABEL_TO_ID["Parasitized"]],
    "correct": test_true == test_predicted,
})
test_results.to_csv(OUTPUT_DIR / "test_predictions.csv", index=False)
report = classification_report(test_true, test_predicted, target_names=CLASS_NAMES,
                               output_dict=True, zero_division=0)
report_df = pd.DataFrame(report).transpose()
display(report_df.round(4))
report_df.to_csv(OUTPUT_DIR / "classification_report.csv")
test_auc = auc(*roc_curve(test_true, test_probabilities[:, LABEL_TO_ID["Parasitized"]])[:2])
test_ap = average_precision_score(test_true, test_probabilities[:, LABEL_TO_ID["Parasitized"]])
print(f"Best validation epoch: {best_epoch}")
print(f"Test ROC-AUC: {test_auc:.4f}; test average precision: {test_ap:.4f}")

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
cm = confusion_matrix(test_true, test_predicted, labels=range(len(CLASS_NAMES)))
ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES).plot(ax=axes[0], cmap="Blues", colorbar=False)
axes[0].set_title("Test confusion matrix")
fpr, tpr, _ = roc_curve(test_true, test_probabilities[:, LABEL_TO_ID["Parasitized"]])
axes[1].plot(fpr, tpr, label=f"AUC = {test_auc:.3f}")
axes[1].plot([0, 1], [0, 1], "--", color="gray")
axes[1].set(title="ROC curve (Parasitized)", xlabel="False positive rate", ylabel="True positive rate")
axes[1].legend(loc="lower right")
precision, recall, _ = precision_recall_curve(test_true, test_probabilities[:, LABEL_TO_ID["Parasitized"]])
axes[2].plot(recall, precision, label=f"AP = {test_ap:.3f}")
axes[2].set(title="Precision-recall curve", xlabel="Recall", ylabel="Precision", xlim=(0, 1), ylim=(0, 1.02))
axes[2].legend(loc="lower left")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "test_metrics.png", dpi=160, bbox_inches="tight")
plt.show()

## 9. Manually check random held-out test images

After training and loading the best validation checkpoint, this cell randomly selects images from the held-out test split, runs predictions, prints the actual/predicted labels and confidence, and saves an image grid plus a CSV. It does not use these examples to tune the model. Re-run the cell for another random sample.

In [ ]:
RANDOM_TEST_SAMPLE_SIZE = min(12, len(test_df))
RANDOM_TEST_SEED = None  # Set an integer (for example 123) to reproduce the same sample.
sample_seed = RANDOM_TEST_SEED if RANDOM_TEST_SEED is not None else random.SystemRandom().randint(0, 2**32 - 1)
random_test_df = test_df.sample(n=RANDOM_TEST_SAMPLE_SIZE, random_state=sample_seed).reset_index(drop=True)
random_test_loader = DataLoader(
    BloodCellDataset(random_test_df, eval_transform),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
)

model.eval()
sample_probabilities = []
with torch.no_grad():
    for images, _, _ in random_test_loader:
        batch_probabilities = torch.softmax(model(images.to(DEVICE)), dim=1).cpu().numpy()
        sample_probabilities.extend(batch_probabilities.tolist())

sample_probabilities = np.asarray(sample_probabilities)
sample_predictions = sample_probabilities.argmax(axis=1)
random_test_results = random_test_df[["path", "label"]].rename(columns={"label": "actual"}).copy()
random_test_results["predicted"] = [CLASS_NAMES[i] for i in sample_predictions]
random_test_results["confidence"] = sample_probabilities.max(axis=1)
random_test_results["prob_parasitized"] = sample_probabilities[:, LABEL_TO_ID["Parasitized"]]
random_test_results["correct"] = random_test_results["actual"] == random_test_results["predicted"]
random_test_results.to_csv(OUTPUT_DIR / "random_test_predictions.csv", index=False)
print(f"Random test sample seed: {sample_seed}")
display(random_test_results.drop(columns="path").round(4))
print(f"Sample accuracy: {random_test_results['correct'].mean():.1%} "
      f"({random_test_results['correct'].sum()}/{len(random_test_results)} images)")

columns = 4
rows = int(np.ceil(len(random_test_results) / columns))
fig, axes = plt.subplots(rows, columns, figsize=(14, 3.6 * rows))
axes = np.asarray(axes).reshape(-1)
for ax, row in zip(axes, random_test_results.itertuples()):
    with Image.open(row.path) as image:
        ax.imshow(image.convert("RGB"))
    ax.set_title(
        f"Actual: {row.actual}\nPredicted: {row.predicted} ({row.confidence:.1%})",
        color="green" if row.correct else "red",
        fontsize=9,
    )
    ax.axis("off")
for ax in axes[len(random_test_results):]:
    ax.axis("off")
fig.suptitle("Random images from the held-out test set")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "random_test_predictions.png", dpi=160, bbox_inches="tight")
plt.show()

## 10. Visual review of test predictions

The grids show representative correct and incorrect test predictions with the model's predicted-class confidence. These examples are for visual review only and do not replace expert microscopy.

In [ ]:
def show_prediction_examples(frame, title, limit=12):
    if frame.empty:
        print(f"No examples: {title}")
        return
    examples = frame.sample(min(limit, len(frame)), random_state=SEED).reset_index(drop=True)
    columns = 4
    rows = int(np.ceil(len(examples) / columns))
    fig, axes = plt.subplots(rows, columns, figsize=(14, 3.4 * rows))
    axes = np.asarray(axes).reshape(-1)
    for ax, row in zip(axes, examples.itertuples()):
        with Image.open(row.path) as image:
            ax.imshow(image.convert("RGB"))
        confidence = max(row.prob_uninfected, row.prob_parasitized)
        ax.set_title(f"Actual: {row.actual}\nPred: {row.predicted} ({confidence:.1%})", fontsize=9)
        ax.axis("off")
    for ax in axes[len(examples):]:
        ax.axis("off")
    fig.suptitle(title)
    plt.tight_layout()
    filename = "correct_test_examples.png" if "Correct" in title else "misclassified_test_examples.png"
    plt.savefig(OUTPUT_DIR / filename, dpi=160, bbox_inches="tight")
    plt.show()

show_prediction_examples(test_results[test_results.correct], "Correct test predictions")
show_prediction_examples(test_results[~test_results.correct], "Misclassified test predictions")

## 11. Artifacts and summary

Artifacts include the best model checkpoint, three split manifests, per-image test predictions, a separate random test-image sample and prediction grid, classification report, training history, and saved plots. Keep the manifests with the model so evaluation can be reproduced.

In [ ]:
summary = {
    "label_source": "parent directory name (Parasitized / Uninfected)",
    "unique_images": int(len(df)),
    "unlabeled_images_excluded": int(len(unlabeled_paths)),
    "invalid_labeled_images_excluded": int(len(invalid_images)),
    "exact_duplicate_files_removed": int(len(all_image_paths) - len(unlabeled_paths) - len(invalid_images) - len(df)),
    "class_counts": {key: int(value) for key, value in counts.items()},
    "split_images": {key: int(value) for key, value in split_summary.images.items()},
    "best_epoch": int(best_epoch),
    "test_accuracy": float(report["accuracy"]),
    "test_roc_auc": float(test_auc),
    "test_average_precision": float(test_ap),
    "seed": SEED,
    "warning": "Research/demo only; not for diagnosis or treatment decisions."
}
with (OUTPUT_DIR / "summary.json").open("w", encoding="utf-8") as stream:
    json.dump(summary, stream, indent=2)
print(json.dumps(summary, indent=2))
print("\nCreated files:")
for path in sorted(OUTPUT_DIR.iterdir()):
    print(f"- {path.name} ({path.stat().st_size:,} bytes)")